# old

In [1]:
import os
from src.db_client import ClientCreator

host = "10.245.12.58"
host_slurm = "arctrdcn018.rs.gsu.edu"
db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
crop_tensor = False
client_creator = ClientCreator(
    db_host, crop_tensor=crop_tensor
)

db_name = databases = "multimodalSubnetworks"
db_collection = collections = "fbirn_falff"
db_fields = ["image", "gender"]
index_id = 'id'
numcubes = 1
cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]
numvolumes = 4
prefetches = 16

client_creator.set_database(databases)
client_creator.set_collection(collections)
client_creator.set_num_subcubes(numcubes)
client_creator.set_shape(subvolume_shape)

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}



/data/users2/ppopov1/miniconda/envs/catalyst12/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [26]:
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader

client = MongoClient("mongodb://" + db_host + ":27017")
db = client[db_name]
posts = db[db_collection + ".bin"]

train_dataset = MongoDataset(
    [0, 1, 2, 3, 4, 5, 6, 7], 
    funcs["mytransform"],
    None,
    db_fields,
    normalize=unit_interval_normalize,
    id=index_id,
)

train_sampler = DBBatchSampler(train_dataset, batch_size=numvolumes)

collate = (
    funcs["mycollate_full"]
    if shape == 256
    else funcs["mycollate"]
)

train_dataloader = BatchPrefetchLoaderWrapper(
            DataLoader(
                train_dataset,
                sampler=train_sampler,
                collate_fn=collate, # COLLATOR TRANSFORMS DATA USING mindfultensors.mongoloader.mcollate
                pin_memory=True,
                worker_init_fn=funcs["createclient"],
                persistent_workers=True,
                prefetch_factor=2,
                num_workers=4,  # self.prefetches,
                # prefetch_factor=None,
                # num_workers=1,  # self.prefetches,
            ),
            num_prefetches=prefetches,
        )

/data/users2/ppopov1/miniconda/envs/catalyst12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


# new

In [2]:
import os
from src.db_client import ClientCreator

host = "10.245.12.58"
host_slurm = "arctrdcn018.rs.gsu.edu"
db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
crop_tensor = False
client_creator = ClientCreator(
    db_host, crop_tensor=crop_tensor
)

db_name = databases = "MindfulTensors"
db_collection = collections = "fbirn"
db_fields = ["image", "gender"]
index_id = 'id'
numcubes = 1
cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]
numvolumes = 4
prefetches = 16

client_creator.set_database(databases)
client_creator.set_collection(collections)
client_creator.set_num_subcubes(numcubes)
client_creator.set_shape(subvolume_shape)

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}



In [6]:
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader

client = MongoClient("mongodb://" + db_host + ":27017")
db = client[db_name]
posts_bin = db[db_collection + ".bin"]
posts_meta = db[db_collection + ".meta"]

In [8]:
posts_meta.find_one(sort=[('id', -1)])

{'_id': ObjectId('69278c3f3eb52c6b06019d82'),
 'id': 356,
 'subject_id': '002058634127',
 'modalities': ['falff', 'smri'],
 'gender': 'M',
 'gender_encoded': 0,
 'modality_info': {'falff': {'data_types': {'image': 'Normalized (0-255 uint8)',
    'label': 'Binary (0: Male, 1: Female, uint8)'},
   'normalization': 'Quantile (0.02-0.98)',
   'chunk_size_mb': 10,
   'num_chunks': 2},
  'smri': {'data_types': {'image': 'Normalized (0-255 uint8)',
    'label': 'Binary (0: Male, 1: Female, uint8)'},
   'normalization': 'Quantile (0.02-0.98)',
   'chunk_size_mb': 10,
   'num_chunks': 2}}}

In [10]:
bin_data = posts_bin.find_one(sort=[('id', -1)])
bin_data["id"], bin_data.keys()

(356, dict_keys(['_id', 'id', 'subject_id', 'chunk_id', 'chunk', 'kind']))

# what to do next
- Figure out how batches are made, modify the label fetching code to use meta collection